<a href="https://colab.research.google.com/github/orz21111/Image-agent-colab/blob/main/Copy_of_Try_redfire_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# התקנות בסיסיות מעודכנות עם קיבוע גרסאות לתאימות
!pip install gradio==4.44.0 transformers torch torchvision diffusers accelerate safetensors faiss-cpu clip huggingface_hub==0.24.1 -q
# התקנת llama-cpp מגרסה מוכנה מראש (Pre-compiled Wheel) למניעת שגיאות בנייה
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q
# התקנות עבור שיפור איכות (Enhancement)
!pip install realesrgan gfpgan -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.24.1 which is incompatible.


In [ ]:
!pip install --upgrade huggingface_hub transformers diffusers accelerate -q

In [ ]:
import os
# STEP 1: Run this to restart the session and clear the import error memory
try:
    import huggingface_hub
    if huggingface_hub.__version__ != '0.24.1':
        print('Restarting session to apply correct library versions...')
        os.kill(os.getpid(), 9)
except:
    pass

import torch
import gc
import requests
import numpy as np
import faiss
import gradio as gr
from PIL import Image
from io import BytesIO
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image, DPMSolverMultistepScheduler
from llama_cpp import Llama
import clip
from realesrgan import RealESRGANer
from gfpgan import GFPGANer

# הגידורי מכשיר וזיכרון
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16

def flush_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("טוען מודל FireRed (המוח)...")
if not os.path.exists("firered.gguf"):
    !wget -q https://huggingface.co/vantagewithai/FireRed-Image-Edit-1.1-GGUF/resolve/main/FireRed-Image-Edit-1.1-Q4_K_M.gguf -O firered.gguf
llm = Llama(model_path="firered.gguf", n_gpu_layers=-1, n_ctx=2048, verbose=False)

print("טוען מודל CLIP (מקודד RAG)...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device="cpu")

vector_dim = 512
index = faiss.IndexFlatIP(vector_dim)
style_database = []

def agent1_learn_styles(file_paths):
    global index, style_database
    if not file_paths: return "לא הועלו קבצים."
    flush_memory()
    clip_model.to(device)
    count = 0
    for path in file_paths:
        try:
            img = Image.open(path).convert("RGB")
            img_input = clip_preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad():
                emb = clip_model.encode_image(img_input).cpu().numpy().astype('float32')
            faiss.normalize_L2(emb)
            index.add(emb)
            style_database.append({"path": path, "name": os.path.basename(path)})
            count += 1
        except Exception as e:
            print(f"שגיאה בעיבוד {path}: {e}")
    clip_model.to("cpu")
    flush_memory()
    return f"הסוכן למד {count} סגנונות חדשים!"

def retrieve_best_style(query_img):
    if len(style_database) == 0: return None
    clip_model.to(device)
    img_input = clip_preprocess(query_img).unsqueeze(0).to(device)
    with torch.no_grad():
        query_emb = clip_model.encode_image(img_input).cpu().numpy().astype('float32')
    faiss.normalize_L2(query_emb)
    D, I = index.search(query_emb, 1)
    best_match_index = I[0][0]
    clip_model.to("cpu")
    return style_database[best_match_index]

def agent3_create(prompt, init_img=None, steps=4, strength=0.7):
    flush_memory()
    model_id = "stabilityai/sdxl-turbo"
    if init_img:
        pipe = AutoPipelineForImage2Image.from_pretrained(model_id, torch_dtype=dtype, variant="fp16").to(device)
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        result = pipe(prompt=prompt, image=init_img, strength=strength, guidance_scale=0.0, num_inference_steps=steps).images[0]
    else:
        pipe = AutoPipelineForText2Image.from_pretrained(model_id, torch_dtype=dtype, variant="fp16").to(device)
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        result = pipe(prompt=prompt, guidance_scale=0.0, num_inference_steps=steps).images[0]
    del pipe
    flush_memory()
    return result

def run_creative_agent(instruction, ref_files, use_rag, steps, realism_lora):
    flush_memory()
    has_refs = ref_files is not None and len(ref_files) > 0
    init_img = None
    augmented_prompt = instruction
    if has_refs:
        init_img = Image.open(ref_files[0]).convert("RGB").resize((512, 512))
        if use_rag and len(style_database) > 0:
            best_style = retrieve_best_style(init_img)
            if best_style:
                prompt_query = f"User wants: {instruction}. Reference style: '{best_style['name']}'. Task: Realistic prompt blending user wish and style."
                response = llm(prompt_query, max_tokens=128)
                augmented_prompt = response['choices'][0]['text'].strip()
    elif not has_refs or (has_refs and not use_rag):
        prompt_query = f"User instruction: {instruction}. Task: Rewrite into high-quality realistic photography prompt."
        response = llm(prompt_query, max_tokens=96)
        augmented_prompt = response['choices'][0]['text'].strip()
    if realism_lora:
        augmented_prompt += ", hyper-realistic, 8k uhd, professional lighting"
    strength = 0.55 if use_rag else 0.75
    result = agent3_create(augmented_prompt, init_img, steps, strength)
    return result, augmented_prompt

with gr.Blocks(theme=gr.themes.Soft(), title="Multi-Agent RAG Learning") as demo:
    gr.Markdown("# 🤖 Creative Agent Network")
    with gr.Row():
        with gr.Column():
            style_files = gr.File(file_count="multiple", label="Learn Styles")
            train_btn = gr.Button("Learn 📖")
            rag_status = gr.Textbox(label="Status")
            prompt_input = gr.Textbox(label="Instruction")
            ref_input = gr.File(file_count="multiple", label="Reference")
            use_rag = gr.Checkbox(label="Use RAG", value=True)
            realism_lora = gr.Checkbox(label="Enhance Realism", value=True)
            steps_slider = gr.Slider(1, 10, 4, step=1, label="Steps")
            run_btn = gr.Button("Create 🚀")
        with gr.Column():
            output_image = gr.Image(type="pil", label="Result")
            refined_prompt_out = gr.Textbox(label="Refined Prompt")
    train_btn.click(agent1_learn_styles, inputs=[style_files], outputs=[rag_status])
    run_btn.click(run_creative_agent, inputs=[prompt_input, ref_input, use_rag, steps_slider, realism_lora], outputs=[output_image, refined_prompt_out])

if __name__ == "__main__":
    demo.launch(share=True, debug=True)